# Clima de Medellín, 2021

Abrí `muestras/open-meteo-medellin-2021.json`. Quería ver si el año está completo, hora por hora, y si el archivo trae el código del municipio.


In [1]:
import json
from pathlib import Path
import pandas as pd

ruta = Path("..") / "muestras" / "open-meteo-medellin-2021.json"
crudo = json.loads(ruta.read_text(encoding="utf-8"))
list(crudo.keys())


['latitude',
 'longitude',
 'generationtime_ms',
 'utc_offset_seconds',
 'timezone',
 'timezone_abbreviation',
 'elevation',
 'hourly_units',
 'hourly']

In [2]:
{
    "latitud": crudo["latitude"],
    "longitud": crudo["longitude"],
    "elevacion": crudo["elevation"],
    "zona": crudo["timezone"],
    "unidades": crudo["hourly_units"],
}


{'latitud': 6.2214413,
 'longitud': -75.55185,
 'elevacion': 1475.0,
 'zona': 'America/Bogota',
 'unidades': {'time': 'iso8601',
  'temperature_2m': '°C',
  'precipitation': 'mm',
  'relative_humidity_2m': '%',
  'wind_speed_10m': 'km/h',
  'surface_pressure': 'hPa'}}

In [3]:
clima = pd.DataFrame(crudo["hourly"])
clima["time"] = pd.to_datetime(clima["time"])
clima.shape


(8760, 6)

In [4]:
clima.head(3)


,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,surface_pressure
0,2021-01-01 00:00:00,17.7,0.1,100,2.3,857.3
1,2021-01-01 01:00:00,17.8,0.1,100,4.0,857.4
2,2021-01-01 02:00:00,17.7,0.1,100,3.6,857.0


In [5]:
clima.isna().sum()


time                    0
temperature_2m          0
precipitation           0
relative_humidity_2m    0
wind_speed_10m          0
surface_pressure        0
dtype: int64

In [6]:
clima["time"].min(), clima["time"].max(), clima["time"].nunique(), clima["time"].duplicated().sum()


(Timestamp('2021-01-01 00:00:00'),
 Timestamp('2021-12-31 23:00:00'),
 8760,
 np.int64(0))

In [7]:
clima.groupby(clima["time"].dt.month).size()


time
1     744
2     672
3     744
4     720
5     744
6     720
7     744
8     744
9     720
10    744
11    720
12    744
dtype: int64

In [8]:
clima[["temperature_2m", "precipitation"]].agg(["min", "max", "mean"])


,temperature_2m,precipitation
min,12.500000,0.000000
max,28.100000,14.300000
mean,19.706929,0.257763


Encontré 8.760 horas, de `2021-01-01 00:00` a `2021-12-31 23:00`. No hay horas repetidas ni celdas vacías. Enero trae 744 horas, que son los 31 días. Febrero trae 672, porque 2021 no es bisiesto.

El punto que guardó la API es latitud 6,2214413, longitud -75,55185, elevación 1.475 metros, zona `America/Bogota`. No es la cabecera del DIVIPOLA. Es el cruce de grilla más cercano. La primera hora mide 17,7 °C y 0,1 mm de lluvia.

La temperatura del año va de 12,5 °C a 28,1 °C. La lluvia suma unos 2.258 mm. Las unidades son °C, mm, % de humedad, km/h de viento y hPa.

El archivo no trae `cod_divipola` ni el nombre de Medellín. El grano es una hora de esa cabecera. Cuando lo cargue le voy a escribir `05001`. No sirve para otro municipio ni para un equipo.
